### Load source and dependencies
**Process:** Extract sql-8b-source.zip and install the training dependencies. Upload the archive when prompted.

**Expected output:** Installation messages stream below the cell. The source folder is created and command logs are written under artifacts/logs.


In [ ]:
from pathlib import Path
import os,sys,subprocess,json,zipfile,shutil
WORK=Path("/kaggle/working" if Path("/kaggle/working").exists() else "/content")
os.chdir(WORK)
candidates=list(WORK.glob("sql-8b-source.zip"))+list(Path("/kaggle/input").rglob("sql-8b-source.zip")) if Path("/kaggle/input").exists() else list(WORK.glob("sql-8b-source.zip"))
if not candidates:
    from google.colab import files
    files.upload()
    candidates=list(WORK.glob("sql-8b-source.zip"))
if len(candidates)!=1: raise RuntimeError("Provide exactly one sql-8b-source.zip")
with zipfile.ZipFile(candidates[0]) as z:
    for name in z.namelist():
        if not (WORK/name).resolve().is_relative_to(WORK.resolve()): raise ValueError("Unsafe zip path")
    z.extractall(WORK)
ROOT=WORK/"sql-8b-source";os.chdir(ROOT)
def run(*args):
    import time
    logdir=ROOT/"artifacts/logs";logdir.mkdir(parents=True,exist_ok=True)
    logpath=logdir/(str(time.time_ns())+".log")
    print("Starting:",args,"Log:",logpath,flush=True)
    with logpath.open("w",encoding="utf-8") as log:
        with subprocess.Popen([sys.executable,"-u",*args],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                              text=True,bufsize=1,env={**os.environ,"PYTHONUNBUFFERED":"1"}) as process:
            for line in process.stdout:
                print(line,end="",flush=True);log.write(line);log.flush()
            code=process.wait()
    if code: raise RuntimeError(f"Command failed ({code}); inspect {logpath}")
run("-m","pip","install","-r","requirements-ml.txt","bitsandbytes>=0.45,<1")


### Check GPU and download model
**Process:** Check CUDA memory, record package versions, and download a pinned Qwen3-8B snapshot.

**Expected output:** GPU details, download progress, artifacts/environment.json, and artifacts/model_revision.json.


In [ ]:
import torch,platform,importlib.metadata as metadata
if not torch.cuda.is_available(): raise RuntimeError("Select a GPU runtime before continuing")
info={"gpu":torch.cuda.get_device_name(0),"vram_gib":torch.cuda.get_device_properties(0).total_memory/2**30,
      "python":platform.python_version(),"packages":{k:metadata.version(k) for k in ["torch","transformers","peft","datasets","bitsandbytes"]}}
Path("artifacts").mkdir(exist_ok=True)
Path("artifacts/environment.json").write_text(json.dumps(info,indent=2))
print(info)
if info["vram_gib"]<14: raise RuntimeError("This configuration needs approximately 16 GB or more GPU memory")
from huggingface_hub import HfApi,snapshot_download
MODEL_ID="Qwen/Qwen3-8B"
MODEL_REVISION=HfApi().model_info(MODEL_ID).sha
MODEL=snapshot_download(MODEL_ID,revision=MODEL_REVISION,ignore_patterns=["*.gguf","*.msgpack","*.h5"])
Path("artifacts/model_revision.json").write_text(json.dumps({"id":MODEL_ID,"revision":MODEL_REVISION},indent=2))


### Prepare dataset
**Process:** Build database-disjoint training, validation and evaluation splits from Spider and validate the reference queries.

**Expected output:** Split counts and artifacts/spider/manifest.json. This uses public development data, not the hidden official test set.


In [ ]:
run("-m","experiments.spider","prepare","--model",MODEL,"--output","artifacts/spider",
    "--train-limit","1000","--val-limit","64","--test-limit","100","--max-length","2048")
print(Path("artifacts/spider/manifest.json").read_text())


### Evaluate the base model
**Process:** Generate one SQL prediction per evaluation question, then compare execution results against the references.

**Expected output:** Per-question progress, base-predictions.jsonl, and base-score.json with execution-match scores and failures.


In [ ]:
run("-m","experiments.spider","predict","--model",MODEL,"--test-file","artifacts/spider/test.jsonl","--output","artifacts/base-predictions.jsonl")
run("-m","experiments.spider","score","--test-file","artifacts/spider/test.jsonl","--predictions","artifacts/base-predictions.jsonl","--output","artifacts/base-score.json")


### Train and save checkpoints
**Process:** Mount your Google Drive, preserve existing evidence, and train up to 100 optimizer steps. Save full checkpoints every 10 steps; resume the latest complete checkpoint when available.

**Expected output:** TRAIN_PROGRESS lines with step, elapsed time and loss; CHECKPOINT_SAVED messages. Checkpoints are stored in MyDrive/ai-project-experiments/sql-8b-run-001/adapter. You complete Drive authorization yourself.


In [ ]:
# Mount persistent checkpoint storage before restarting training.
# Complete Colab's authorization yourself; no credentials are shared with the agent.
from google.colab import drive
drive.mount("/content/drive")
PERSIST=Path("/content/drive/MyDrive/ai-project-experiments/sql-8b-run-001")
PERSIST.mkdir(parents=True,exist_ok=True)
# Preserve current evidence without overwriting an earlier saved copy.
for name in ["environment.json","model_revision.json","base-predictions.jsonl","base-score.json"]:
    src=ROOT/"artifacts"/name;dst=PERSIST/name
    if src.exists() and not dst.exists(): shutil.copy2(src,dst)
if not (PERSIST/"spider").exists(): shutil.copytree(ROOT/"artifacts/spider",PERSIST/"spider")
ADAPTER=str(PERSIST/"adapter")
MAX_STEPS=100
completed=sorted((PERSIST/"adapter").glob("checkpoint-*"),
    key=lambda p:int(p.name.split("-")[-1]) if p.name.split("-")[-1].isdigit() else -1)
completed=[p for p in completed if (p/"complete.json").exists() and (p/"trainer_state.json").exists()]
if completed and json.loads((completed[-1]/"trainer_state.json").read_text())["global_step"]>=MAX_STEPS:
    # Validate this run's inputs before reusing a completed checkpoint.
    from training.checkpoints import file_hash
    saved=json.loads((PERSIST/"adapter/training_identity.json").read_text())
    assert saved["train"]==file_hash("artifacts/spider/train.jsonl")
    assert saved["validation"]==file_hash("artifacts/spider/validation.jsonl")
    assert saved["base"]==file_hash(Path(MODEL)/"config.json")
    ADAPTER=str(completed[-1])
    print("Training already reached the target. Using:",ADAPTER,flush=True)
else:
    run("-m","training.train","--model_id",MODEL,"--train_file","artifacts/spider/train.jsonl",
        "--val_file","artifacts/spider/validation.jsonl","--output_dir",ADAPTER,
        "--batch_size","1","--gradient_accumulation_steps","8","--max_seq_length","2048",
        "--max_steps",str(MAX_STEPS),"--epochs","1","--report_to","none",
        "--save_steps","10","--resume_from_checkpoint","auto")


### Evaluate the adapter
**Process:** Generate predictions using the saved adapter on the same evaluation questions, then score and compare against the base model.

**Expected output:** Per-question progress, adapter-predictions.jsonl, adapter-score.json, and base/adapter scores with confidence intervals.


In [ ]:
run("-m","experiments.spider","predict","--model",MODEL,"--adapter",ADAPTER,
    "--test-file","artifacts/spider/test.jsonl","--output","artifacts/adapter-predictions.jsonl")
run("-m","experiments.spider","score","--test-file","artifacts/spider/test.jsonl",
    "--predictions","artifacts/adapter-predictions.jsonl","--output","artifacts/adapter-score.json")
for label in ["base","adapter"]:
    r=json.loads(Path(f"artifacts/{label}-score.json").read_text())
    print(label,{k:r[k] for k in ["num_samples","accuracy_pct","wilson_95_pct","nonempty_accuracy_pct","reference_errors"]})


### Save experiment evidence
**Process:** Package model provenance, splits, SQLite fixtures, predictions, scores and adapter checkpoints. Copy the archive to Google Drive and download it.

**Expected output:** sql-8b-results.zip in runtime storage and MyDrive/ai-project-experiments/sql-8b-run-001. Single-database execution matching is not official Spider test-suite accuracy.


In [ ]:
# Include the small SQLite fixtures so evidence remains independently executable.
for split in ["train","validation","test"]:
    f=Path(f"artifacts/spider/{split}.jsonl")
    records=[json.loads(line) for line in f.read_text().splitlines()]
    for rec in records:
        dest=Path("artifacts/spider/databases")/(rec["db_id"]+".sqlite")
        dest.parent.mkdir(exist_ok=True)
        if not dest.exists(): shutil.copy2(rec["database"],dest)
        rec["database"]=str(dest)
    f.write_text("\n".join(json.dumps(r) for r in records)+"\n")
manifest=Path("artifacts/spider/manifest.json")
data=json.loads(manifest.read_text())
data["original_files_sha256"]=data["files_sha256"]
import hashlib
data["files_sha256"]={s:hashlib.sha256(Path(f"artifacts/spider/{s}.jsonl").read_bytes()).hexdigest() for s in ["train","validation","test"]}
data["database_files_sha256"]={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in Path("artifacts/spider/databases").glob("*.sqlite")}
manifest.write_text(json.dumps(data,indent=2))
# Include the persistent trained adapter in the downloadable evidence.
if (ROOT/"artifacts/adapter").resolve()!=Path(ADAPTER).resolve():
    shutil.copytree(ADAPTER,ROOT/"artifacts/adapter",dirs_exist_ok=True)
archive=shutil.make_archive(str(WORK/"sql-8b-results"),"zip",ROOT,"artifacts")
shutil.copy2(archive,PERSIST/"sql-8b-results.zip")
print("Results:",archive,"Persistent copy:",PERSIST/"sql-8b-results.zip")
if str(WORK)=="/content":
    from google.colab import files
    files.download(archive)
